<a href="https://colab.research.google.com/github/szhang2187/Datathon_For_Social_Good_8/blob/data_cleaning_step-1/01_clean_places.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning

In [ ]:
# import pandas, numpy
import pandas as pd
import numpy as np

In [ ]:
# import dataset
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# sample 1000 rows of google map places dataset

path = '/content/drive/MyDrive/Copy of google_maps.csv'
df_peek = pd.read_csv(path, nrows=1000)   # nrows = only read the first 1,000 rows
print(df_peek.columns.tolist())
df_peek.head(5)

# keep relevant columns
cols = ['place_id', 'category', 'rating', 'reviews_count', 'name', 'address','permanently_closed']

['place_id', 'url', 'country', 'name', 'category', 'address', 'description', 'business_details', 'open_hours', 'reviews_count', 'rating', 'main_image', 'reviews', 'lat', 'lon', 'services_provided', 'hotel_amenities', 'hotel_star_ratings', 'open_website', 'phone_number', 'permanently_closed', 'photos_and_videos', 'people_also_search', 'web_results', 'reservation_link', 'questions_answers', 'top_reviews', 'reviews_snippets', 'directory_categories', 'directory_locations', 'popular_times', 'cid_location', 'is_claimed', 'fid_location', 'review_distribution', 'all_categories', 'country_code', 'cid', 'temporarily_closed', 'category_index', 'about']


In [ ]:
# load full file, then clean and log each step
df = pd.read_csv(path, usecols=cols)
drop_log = [("start (raw file)", len(df))]

print(df["permanently_closed"].value_counts(dropna=False))   # see True/False/missing first

# 1. closed places
before = len(df)
df = df[df["permanently_closed"] == False]
drop_log.append(("dropped closed", before - len(df)))

# 2. duplicate place_ids
before = len(df)
df = df.drop_duplicates(subset="place_id")
drop_log.append(("dropped duplicate place_id", before - len(df)))

# 3. no address
before = len(df)
df = df[df["address"].fillna("").str.strip() != ""]
drop_log.append(("dropped no address", before - len(df)))

for step, n in drop_log:
    print(step, n)
print("final rows:", len(df))

permanently_closed
False    135262
True      12167
Name: count, dtype: int64
start (raw file) 147429
dropped closed 12167
dropped duplicate place_id 0
dropped no address 1441
final rows: 133821


# Data Stats + more analysis

In [ ]:
df.info()                  # column types + how many cells are filled

<class 'pandas.core.frame.DataFrame'>
Index: 133821 entries, 0 to 147428
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   place_id            133821 non-null  object 
 1   name                133819 non-null  object 
 2   category            133821 non-null  object 
 3   address             133821 non-null  object 
 4   reviews_count       133806 non-null  float64
 5   rating              133821 non-null  float64
 6   permanently_closed  133821 non-null  bool   
dtypes: bool(1), float64(2), object(4)
memory usage: 7.3+ MB


In [ ]:
df.isna().sum()            # missing values per column

,0
place_id,0
name,2
category,0
address,0
reviews_count,15
rating,0
permanently_closed,0


In [ ]:
df["place_id"].nunique()   # how many unique places

133821

In [ ]:
# A min of 0 or a max above 5 means bad or placeholder values to fix before analysis.
df["rating"].describe()

,rating
count,133821.000000
mean,1.812163
std,2.219250
min,0.000000
25%,0.000000
50%,0.000000
75%,4.500000
max,5.000000


In [ ]:
# Shows which labels to map to our 7 service types in category_map.csv,
# and which off-topic ones (e.g. "skin care clinic") to exclude.
df["category"].value_counts().head(20)

,count
category,
Bus stop,18261
Doctor,11244
Dentist,8390
Medical clinic,4987
Internist,4785
Psychologist,4491
Non-profit organization,3869
Transport hub,3429
Psychotherapist,3367


In [ ]:
# CHECK: do 0 ratings line up with places that have no reviews?
print((df["rating"] == 0).sum())
print(((df["rating"] == 0) & (df["reviews_count"] == 0)).sum())

77203
77188


In [ ]:
# CLEAN: treat 0 as "no rating" (NaN = missing), keeping the original column untouched
df["rating_clean"] = df["rating"].replace(0, np.nan)
print(df["rating_clean"].describe())

count    56618.000000
mean         4.283187
std          1.028035
min          1.000000
25%          4.000000
50%          4.800000
75%          5.000000
max          5.000000
Name: rating_clean, dtype: float64


### Extract ZIPS

In [ ]:
# CHECK: what do real addresses look like? (random_state makes the sample repeatable)
print(df["address"].sample(8, random_state=42).to_string())

4876                    San Rafael, CA 94903, United States
89335                    Los Altos, CA 94024, United States
129531    1040 Lincoln Ave, San Jose, CA 95125, United S...
111454    1650 Las Plumas Ave, San Jose, CA 95133, Unite...
17682                      Hayward, CA 94544, United States
111890    20126 Stanton Ave, Castro Valley, CA 94546, Un...
15527     500 Alfred Nobel Dr Ste 180, Hercules, CA 9454...
61880     280 W MacArthur Blvd, Oakland, CA 94611, Unite...


In [ ]:
# CLEAN: pull the 5-digit ZIP that follows "CA" or "California".
# Also handles ZIP+4 (94070-2603 or 940702603): we keep only the first 5 digits.
# (?:...) = group we don't keep, | = "or", -? = optional hyphen, \s+ = one or more spaces
df["zip"] = df["address"].str.extract(r'\b(?:CA|California)\s+(\d{5})(?:-?\d{4})?\b')

# CHECK: how many rows failed to get a ZIP?
n_fail = df["zip"].isna().sum()
print(n_fail, n_fail / len(df) * 100)    # count and percent

# CHECK: what do the remaining failures look like?
print(df.loc[df["zip"].isna(), "address"].sample(10, random_state=42).to_string())

# CHECK: ZIP prefixes (Bay Area should be almost all 94 and 95)
print(df["zip"].str[:2].value_counts())

361 0.269763340581822
67026                           Berkeley, CA, United States
10731                             California, United States
4552                              California, United States
129359                          San Jose, CA, United States
20217                           San Jose, CA, United States
73535                           Piedmont, CA, United States
27305     10175 SW Barbur Blvd #300bc, Portland, OR 9721...
45016                      San Francisco, CA, United States
109349                          San Jose, CA, United States
46208     119 Hawthorne Ave, Camden, AL 36726, United St...
zip
94    99941
95    33499
91        5
10        3
93        2
92        2
70        1
78        1
90        1
59        1
65        1
31        1
98        1
32        1
Name: count, dtype: int64


In [ ]:
# CHECK: what do the 361 remaining failures look like?
print(df.loc[df["zip"].isna(), "address"].sample(10, random_state=42).to_string())

# CHECK: ZIP prefixes (Bay Area should be almost all 94 and 95)
print(df["zip"].str[:2].value_counts())

67026                           Berkeley, CA, United States
10731                             California, United States
4552                              California, United States
129359                          San Jose, CA, United States
20217                           San Jose, CA, United States
73535                           Piedmont, CA, United States
27305     10175 SW Barbur Blvd #300bc, Portland, OR 9721...
45016                      San Francisco, CA, United States
109349                          San Jose, CA, United States
46208     119 Hawthorne Ave, Camden, AL 36726, United St...
zip
94    99941
95    33499
91        5
10        3
93        2
92        2
70        1
78        1
90        1
59        1
65        1
31        1
98        1
32        1
Name: count, dtype: int64


In [ ]:
# CLEAN: drop places with no ZIP in the address (can't join to Census data without one)
before = len(df)
df = df[df["zip"].notna()]
drop_log.append(("dropped no ZIP in address", before - len(df)))

for step, n in drop_log:
    print(step, n)
print("final rows:", len(df))

start (raw file) 147429
dropped closed 12167
dropped duplicate place_id 0
dropped no address 1441
dropped no ZIP in address 361
final rows: 133460


## Building Categories

In [ ]:
# CHECK: how many distinct category labels, and the top 60 (we map these to service types)
print(df["category"].nunique())
print(df["category"].value_counts().head(60).to_string())

393
category
Bus stop                              18258
Doctor                                11243
Dentist                                8388
Medical clinic                         4986
Internist                              4785
Psychologist                           4491
Non-profit organization                3868
Psychotherapist                        3364
Pediatrician                           3187
Transport hub                          3094
Family practice physician              2815
Psychiatrist                           2615
Marriage or relationship counselor     2231
Optometrist                            2051
Preschool                              2048
Day care center                        1904
Obstetrician-gynecologist              1796
Money transfer service                 1764
Grocery store                          1663
Acupuncture clinic                     1618
Mental health service                  1576
Physical therapist                     1502
Bank               

In [ ]:
# CHECK: labels (any rank) containing keywords tied to our 7 service types
keywords = "pharm|drug|grocer|supermarket|market|food|pantry|urgent|hospital|clinic|school|bank|credit union|bus"
labels = df["category"].value_counts()
print(labels[labels.index.str.contains(keywords, case=False)].to_string())

category
Bus stop                                 18258
Medical clinic                            4986
Preschool                                 2048
Grocery store                             1663
Acupuncture clinic                        1618
Bank                                      1461
Pharmacy                                  1163
Elementary school                         1156
Martial arts school                        794
Dance school                               744
Mental health clinic                       688
High school                                592
Dental clinic                              586
Physical therapy clinic                    476
Music school                               469
After school program                       417
Food bank                                  414
Middle school                              398
Hospital                                   349
Driving school                             284
Art school                                 281
Cred



- We have to hard-code what goes into which category, since written labels can be misleading (eg. 'martial arts school' is not a school)
--

**clinic/hospital:** general care, hospitals, urgent care, dentists, and mental health. Specialists (cardiologist, surgeon, etc.) and elective or alternative care (acupuncture, plastic surgery, fertility) stay out.

**grocery:** grocery stores and supermarkets, including ethnic grocers. Convenience stores stay out.

**school:** K-12 only. Preschool, martial arts, and driving schools stay out.

**bank:** banks and credit unions. Investment banks, blood banks, and sperm banks stay out.

A clinic_subtype column (medical, dental, mental health) lets you split clinics later.




In [ ]:
# CLEAN: map exact category labels to our 7 service types.
# Labels are lowercased first, so "Credit Union" and "Credit union" count as the same thing.
medical = ["doctor", "medical clinic", "clinic", "urgent care center", "urgent care centre",
           "walk-in clinic", "free clinic", "emergency clinic", "emergency care physician",
           "medical center", "hospital", "general hospital", "children's hospital",
           "university hospital", "government hospital", "private hospital",
           "veterans hospital", "military hospital", "maternity hospital",
           "internist", "family practice physician", "pediatrician", "pediatric clinic",
           "obstetrician-gynecologist", "women's health clinic", "children policlinic", "optometrist", "physical therapist", "physical therapy clinic"]
dental = ["dentist", "dental clinic", "pediatric dentist"]
mental = ["psychologist", "psychotherapist", "psychiatrist", "mental health service",
          "mental health clinic", "marriage or relationship counselor", "counselor",
          "family counselor", "mental hospital", "psychiatric hospital", "addiction treatment center"]

groceries = ["grocery store", "supermarket", "asian grocery store", "indian grocery store",
             "mexican grocery store", "gourmet grocery store", "japanese grocery store",
             "korean grocery store", "filipino grocery store", "european grocery store",
             "italian grocery store", "russian grocery store", "kosher grocery store",
             "american grocery store", "eastern european grocery store",
             "chinese supermarket", "japanese supermarket", "discount supermarket",
             "asian grocery shop", "gourmet grocery shop", "indian grocery shop",
             "produce market"]
banks = ["bank", "credit union", "federal credit union", "cooperative bank",
         "trust bank", "private sector bank"]
schools = ["elementary school", "middle school", "high school", "primary school",
           "secondary school", "k-12 school", "charter school", "catholic school",
           "senior high school", "girls' high school", "boys' high school"]

mapping = {}
for label in medical + dental + mental: mapping[label] = "clinic/hospital"
for label in groceries:                 mapping[label] = "grocery"
for label in banks:                     mapping[label] = "bank"
for label in schools:                   mapping[label] = "school"
for label in ["pharmacy", "homeopathic pharmacy"]: mapping[label] = "pharmacy"
for label in ["food bank", "foodbank"]:            mapping[label] = "food bank"
mapping["bus stop"] = "bus stop"

subtype = {}
for label in medical: subtype[label] = "medical"
for label in dental:  subtype[label] = "dental"
for label in mental:  subtype[label] = "mental health"

df["category_lc"] = df["category"].str.lower().str.strip()
df["service_type"] = df["category_lc"].map(mapping)       # no match = NaN
df["clinic_subtype"] = df["category_lc"].map(subtype)

# SAVE: the lookup table for the repo (category_map.csv)
pd.Series(mapping, name="service_type").rename_axis("category_lc").reset_index() \
  .to_csv("category_map.csv", index=False)

# CHECK: places per service type, and how many are left unmapped
print(df["service_type"].value_counts(dropna=False))
print(df["clinic_subtype"].value_counts())

service_type
clinic/hospital    63111
NaN                43751
bus stop           18258
grocery             2546
school              2416
bank                1786
pharmacy            1165
food bank            427
Name: count, dtype: int64
clinic_subtype
medical          35850
mental health    17904
dental            9357
Name: count, dtype: int64


In [ ]:
# CHECK: biggest unmapped labels. Did we skip anything we should include?
unmapped = df.loc[df["service_type"].isna(), "category_lc"].value_counts()
print(unmapped.head(40).to_string())

category_lc
non-profit organization         3868
transport hub                   3094
preschool                       2048
day care center                 1964
money transfer service          1764
acupuncture clinic              1618
convenience store               1368
orthopedic surgeon              1336
employment agency               1289
playground                      1247
city government office          1170
cardiologist                    1002
surgeon                          930
social services organization     911
martial arts school              794
fire station                     770
dance school                     744
podiatrist                       717
county government office         705
laundromat                       699
oncologist                       681
government office                662
family day care service          648
child care agency                602
plastic surgeon                  562
community center                 549
post office               

In [ ]:
# CHECK: how many clinic places share an address with other clinic places?
clin = df[df["service_type"] == "clinic/hospital"]
print(len(clin), clin["address"].nunique())          # places vs. distinct addresses
print(clin["address"].value_counts().head(10).to_string())   # biggest clusters

63111 26207
address
710 Lawrence Expy, Santa Clara, CA 95051, United States      443
505 Parnassus Ave, San Francisco, CA 94143, United States    326
280 W MacArthur Blvd, Oakland, CA 94611, United States       323
747 52nd St, Oakland, CA 94609, United States                279
300 N Pasteur Dr, Stanford, CA 94305, United States          261
2500 Alhambra Ave, Martinez, CA 94553, United States         256
300 N Pasteur Dr, Palo Alto, CA 94304, United States         245
1425 S Main St, Walnut Creek, CA 94596, United States        241
1411 E 31st St, Oakland, CA 94602, United States             203
975 Sereno Dr, Vallejo, CA 94589, United States              202


In [ ]:
# CHECK: what do bus stop addresses look like, and which are repeated most?
bs = df[df["service_type"] == "bus stop"]
print(bs["address"].sample(8, random_state=42).to_string())
print(bs["address"].value_counts().head(5).to_string())

95270         San Ramon, CA 94583, United States
114967         Stanford, CA 94305, United States
13051     San Francisco, CA 94132, United States
18987           Oakland, CA 94621, United States
129165         San Jose, CA 95121, United States
102549      San Leandro, CA 94577, United States
72352          Pacifica, CA 94044, United States
72680     San Francisco, CA 94127, United States
address
South San Francisco, CA 94080, United States    236
San Francisco, CA 94124, United States          213
San Francisco, CA 94112, United States          208
Oakland, CA 94605, United States                192
Union City, CA 94587, United States             188


In [ ]:
for t in df["service_type"].dropna().unique():
    print("\n==", t)
    print(df.loc[df["service_type"] == t, ["name", "category"]].sample(20, random_state=42).to_string())


== clinic/hospital
                                                    name                   category
89539                            Dr. Garrick K. Wang, MD                     Doctor
21487                                 Chavez Dental Care                    Dentist
67082                               Thomas J. Watson, MD                     Doctor
96018                                    MEDx Disruptors             Medical clinic
124008     John "Jaune" Hanig, LMFT - Honey Arts Therapy            Psychotherapist
6800                   Berkeley Therapeutic Laser Clinic             Medical clinic
49891                             Dr. Belinda Magallanes  Family practice physician
139872                               Silver Deborah J MD               Psychiatrist
139426                                      Iva Ilic, MD  Family practice physician
57594                                   White David a MD   Emergency care physician
114242                 Napa Dentist - Deirdre Burke, DDS

In [ ]:
# CHECK: pharmacy names that do NOT contain pharmacy-like words. Real stores or people?
ph = df[df["service_type"] == "pharmacy"].copy()
pat = r"pharm|drug|\brx\b|cvs|walgreens|rite aid|longs|costco|safeway|kaiser|apothec|compound"
ph["looks_like_pharmacy"] = ph["name"].str.contains(pat, case=False, na=False)

print(ph["looks_like_pharmacy"].value_counts())
print(ph.loc[~ph["looks_like_pharmacy"], "name"].head(40).to_string())

looks_like_pharmacy
True     922
False    243
Name: count, dtype: int64
971                                     Rph
1321            California Health Advocates
1523                            Body By Ann
3008                    Heinemann Shirley S
3238                        Laghaei Farshid
3254                            Miller Masy
3643                      Cato Research Ltd
3809                            Hinsz Clyde
4157                   Jamie L. Silzer, BSP
4186                      Widjaja Melinda I
5864                                 bbarsf
7368                         daviea nedical
8514     Contra Costa County MEDS Coalition
8615                    Vishakha Khanna, RD
8751                        Adona Annabelle
10901                       Hirani Munira K
11636                        Lai Jennifer B
12506                Joanne O. Whitney, PHD
13007                      Nguyen Bachyen T
14577                         Glatt Medical
14579                Forestville Mercantile
1519

In [ ]:
# CLEAN: keep only pharmacy listings that are businesses, not individual pharmacists.
# A name counts as a business if it has a pharmacy-like word or is on our keep list.
# Names ending in a credential (", BPharm", ", PharmD", ", RPh") are individuals.
is_ph = df["service_type"] == "pharmacy"
pat = r"pharm|drug|\brx\b|cvs|walgreens|rite aid|longs|costco|safeway|kaiser|apothec|compound"
looks = df["name"].str.contains(pat, case=False, na=False)
keep_list = df["name"].str.lower().str.strip().isin(["genoa healthcare", "direct meds"])
person = df["name"].str.contains(r",\s*(?:b\.?pharm|pharm\.?d|r\.?ph|bsp)\b", case=False, na=False)

not_business = is_ph & ~((looks | keep_list) & ~person)
print("pharmacy listings before:", is_ph.sum(), "| unlabeled:", not_business.sum())
df.loc[not_business, "service_type"] = np.nan
print("pharmacy listings after:", (df["service_type"] == "pharmacy").sum())

pharmacy listings before: 1165 | unlabeled: 254
pharmacy listings after: 911


In [ ]:
# CLEAN: flag one listing per address within each service type = one "site".
# Every row is kept; is_site marks which rows to count when counting distinct sites.
df["address_key"] = df["address"].str.lower().str.strip()
df["is_site"] = ~df.duplicated(subset=["service_type", "address_key"])
df.loc[df["service_type"] == "bus stop", "is_site"] = True     # bus stops share generic addresses; each is its own site
df.loc[df["service_type"].isna(), "is_site"] = False           # unmapped places don't count

# CHECK: listings vs. sites for each service type
print(df.groupby("service_type")["is_site"].agg(listings="size", sites="sum"))

                 listings  sites
service_type                    
bank                 1786   1744
bus stop            18258  18258
clinic/hospital     63111  26170
food bank             427    422
grocery              2546   2473
pharmacy              911    837
school               2416   2268


In [ ]:
print(df.loc[df["service_type"] == "pharmacy", "name"].sample(20, random_state=7).to_string())

98342     Pharmacy l Kaiser Permanente Mission Bay Medic...
66537                                          Pearl Pharma
42776                                       Costco Pharmacy
61749                                  Health Mart Pharmacy
113698                                       Mega Drugs LLC
98872     OPI - New Tiburcio Vasquez Health Center Pharmacy
131764                                   Walgreens Pharmacy
91747                            Visitacion Valley Pharmacy
86719                      Sebastopol Family Pharmacy, Inc.
93572                                          CVS Pharmacy
139746                                   Walgreens Pharmacy
119491                                   Walgreens Pharmacy
134347                                  Lifescript Pharmacy
122037                                     Safeway Pharmacy
11163                                          CVS Pharmacy
55310                                      Walmart Pharmacy
54095                                   

In [ ]:
# CLEAN: standardize names (lowercase, no punctuation) so "CVS Pharmacy" and "CVS  pharmacy!" match
df["name_clean"] = (df["name"].str.lower()
                    .str.replace(r"[^a-z0-9 ]", " ", regex=True)
                    .str.replace(r"\s+", " ", regex=True).str.strip())

# CLEAN: chain = same cleaned name appears 5+ times within the same service type (bus stops excluded)
name_counts = df.groupby(["service_type", "name_clean"])["place_id"].transform("size")
df["is_chain"] = (name_counts >= 5) & df["service_type"].notna() & (df["service_type"] != "bus stop")

# CLEAN: enough reviews to trust the rating? (the plan's cutoff is 10)
df["enough_reviews"] = df["reviews_count"] >= 10

# CHECK: chain counts per type, top 30 chain names, and share of each type with enough reviews
print(df.groupby("service_type")["is_chain"].sum())
print(df.loc[df["is_chain"], "name_clean"].value_counts().head(30).to_string())
print(df.groupby("service_type")["enough_reviews"].mean().round(2))

service_type
bank               1260
bus stop              0
clinic/hospital     561
food bank             0
grocery             692
pharmacy            554
school               26
Name: is_chain, dtype: int64
name_clean
chase bank                                  201
wells fargo bank                            195
cvs pharmacy                                194
safeway                                     166
walgreens pharmacy                          115
bank of america financial center            114
safeway pharmacy                            102
u s bank branch                             100
bmo bank n a branches                        91
citi                                         83
grocery outlet                               66
target grocery                               60
trader joe s                                 55
covid 19 drive thru testing at walgreens     52
lucky                                        47
whole foods market                           42
bank of ame

In [ ]:
# CLEAN: blank or generic names are not chains
generic = ["dentist", "doctor", "clinic", "pharmacy", "market", "bank"]
df.loc[(df["name_clean"] == "") | df["name_clean"].isin(generic), "is_chain"] = False

print(df.groupby("service_type")["is_chain"].sum())

service_type
bank               1260
bus stop              0
clinic/hospital     523
food bank             0
grocery             687
pharmacy            554
school               26
Name: is_chain, dtype: int64


In [ ]:
# CHECK: do schools have review counts at all?
sch = df[df["service_type"] == "school"]
print(sch["reviews_count"].describe())
print(sch["rating"].eq(0).mean())    # share of schools with no rating

count    2416.000000
mean        0.000414
std         0.020345
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: reviews_count, dtype: float64
0.9983443708609272


# Load in Google Maps Reviews file

In [ ]:
rev_path = "/content/drive/MyDrive/Copy of google_maps_reviews.csv"    # your real filename
rev_peek = pd.read_csv(rev_path, nrows=1000)
print(rev_peek.columns.tolist())
rev_peek.head(3)

['url', 'place_id', 'place_name', 'country', 'address', 'review_id', 'reviewer_name', 'reviews_by_reviewer', 'photos_by_reviewer', 'reviewer_url', 'local_guide', 'review_rating', 'review', 'review_date', 'number_of_likes', 'response_of_owner', 'response_date', 'photos', 'days_limit', 'review_details', 'profile_pic_url', 'place_general_rating', 'overall_place_riviews', 'questions_answers', 'fid_location', 'category', 'cid', 'sort_by']


,url,place_id,place_name,country,address,review_id,reviewer_name,reviews_by_reviewer,photos_by_reviewer,reviewer_url,...,days_limit,review_details,profile_pic_url,place_general_rating,overall_place_riviews,questions_answers,fid_location,category,cid,sort_by
0,https://www.google.com/maps/place/?q=place_id:...,ChIJq1JhLaYcE4gRKLSvYC8rhwg,Econo Lodge,US,"4320 South St Building A, Lafayette, IN 47905",ChZDSUhNMG9nS0VJQ0FnSURmNnJYR0hnEAE,"""""",0,NaN,NaN,...,NaN,NaN,https://lh3.googleusercontent.com/local-review...,2.8,242.0,NaN,NaN,NaN,,NaN
1,https://www.google.com/maps/place/?q=place_id:...,ChIJZUwDOPIBw4kRgM86Rq-e5XY,Red Roof Inn Fairfield,US,"286 US-46, Fairfield, NJ 07004",ChZDSUhNMG9nS0VJQ0FnSUQ3cWJDbVZREAE,Alphonse Vaillan,1,2.0,https://www.google.com/maps/contrib/1021257167...,...,NaN,"[{""title"":""Rooms"",""value"":""1""},{""title"":""Servi...",https://lh3.googleusercontent.com/a/ACg8ocLFnb...,4.0,150.0,NaN,NaN,NaN,,NaN
2,https://www.google.com/maps/place/?q=place_id:...,ChIJxekMCpGdwYcR7zq7bF9nG14,Super 8 by Wyndham Richmond,US,"888 Slumber Ln, Richmond, MO 64085",ChdDSUhNMG9nS0VJQ0FnSURiaC0zRXRRRRAB,Patti B,0,NaN,NaN,...,NaN,NaN,https://lh3.googleusercontent.com/local-review...,4.0,311.0,NaN,NaN,NaN,,NaN


In [ ]:
# CLEAN: keep only places that map to one of our 7 service types
before = len(df)
df_clean = df[df["service_type"].notna()].copy()
drop_log.append(("out of scope (no service type)", before - len(df_clean)))

# SAVE: the columns teammates B and C need
keep = ["place_id", "name", "name_clean", "address", "zip", "category", "service_type",
        "clinic_subtype", "rating", "rating_clean", "reviews_count", "enough_reviews",
        "is_chain", "is_site"]
df_clean[keep].to_csv("/content/drive/MyDrive/places_clean.csv", index=False)

# CHECK
for step, n in drop_log:
    print(step, n)
print("saved rows:", len(df_clean))
print(df_clean["service_type"].value_counts())

start (raw file) 147429
dropped closed 12167
dropped duplicate place_id 0
dropped no address 1441
dropped no ZIP in address 361
out of scope (no service type) 44005
saved rows: 89455
service_type
clinic/hospital    63111
bus stop           18258
grocery             2546
school              2416
bank                1786
pharmacy             911
food bank            427
Name: count, dtype: int64


In [ ]:
# CLEAN: read the reviews file in pieces; keep 1-2 star reviews of places in places_clean
rev_path = "/content/drive/MyDrive/Copy of google_maps_reviews.csv"
rev_cols = ["place_id", "review_id", "review_rating", "review", "review_date"]   # no reviewer names
good_ids = set(df_clean["place_id"])

kept, rows_read = [], 0
for chunk in pd.read_csv(rev_path, usecols=rev_cols, chunksize=200_000):
    rows_read += len(chunk)
    chunk["review_rating"] = pd.to_numeric(chunk["review_rating"], errors="coerce")
    chunk = chunk[chunk["place_id"].isin(good_ids) & (chunk["review_rating"] <= 2)]
    kept.append(chunk)

rev_low = pd.concat(kept, ignore_index=True)
print("review rows read:", rows_read)
print("kept (our places, 1-2 stars):", len(rev_low))

review rows read: 213536
kept (our places, 1-2 stars): 646


In [ ]:
# CLEAN: attach zip, service type and chain flag from places_clean
rev_low = rev_low.merge(df_clean[["place_id", "zip", "service_type", "is_chain"]],
                        on="place_id", how="left")

# SAVE for teammate C
rev_low.to_csv("/content/drive/MyDrive/reviews_low.csv", index=False)

# CHECK: counts only (no review text)
print(rev_low["service_type"].value_counts())
print("places with 1+ low reviews:", rev_low["place_id"].nunique())

service_type
clinic/hospital    646
Name: count, dtype: int64
places with 1+ low reviews: 92


In [ ]:
pd.DataFrame(drop_log, columns=["step", "rows"]).to_csv("/content/drive/MyDrive/drop_log.csv", index=False)

In [ ]:
import os, shutil                       # os = work with folders, shutil = copy files

# CLEAN: copy the finished files into the folder layout the plan asks for
out = "/content/drive/MyDrive/datathon/data/processed"   # destination folder (matches the plan's data/processed/)
os.makedirs(out, exist_ok=True)         # create the folders if missing; exist_ok=True means no error if they already exist

for f in ["places_clean.csv", "drop_log.csv", "reviews_low.csv"]:   # the three files to copy, one at a time
    shutil.copy(f"/content/drive/MyDrive/{f}", out)                  # copy file f from the top of your Drive into the new folder

print(os.listdir(out))                  # list what's in the folder; should show all three file names

['places_clean.csv', 'drop_log.csv', 'reviews_low.csv']


In [ ]:
# SAVE: write the category lookup table to the repo folder so the mapping decisions are recorded
pd.Series(mapping, name="service_type") \
  .rename_axis("category_lc") \
  .reset_index() \
  .to_csv("/content/drive/MyDrive/datathon/data/processed/category_map.csv", index=False)

In [ ]:
print(pd.read_csv("/content/drive/MyDrive/datathon/data/processed/category_map.csv").head())   # show the first 5 rows of the saved map

          category_lc     service_type
0              doctor  clinic/hospital
1      medical clinic  clinic/hospital
2              clinic  clinic/hospital
3  urgent care center  clinic/hospital
4  urgent care centre  clinic/hospital


# US Census Bureau

In [ ]:
import requests                          # requests = lets Python ask a website for data

# the Census variable codes from the plan (the E at the end means "estimate")
vars_ = ["B19013_001E",                                  # median household income
         "B01003_001E",                                  # total population
         "B03002_001E", "B03002_003E", "B03002_004E",    # race/ethnicity: total, white, Black
         "B03002_006E", "B03002_012E",                   # race/ethnicity: Asian, Hispanic
         "B08201_001E", "B08201_002E",                   # households: total, with no car
         "C16002_001E", "C16002_004E", "C16002_007E",    # households: total, then limited-English groups
         "C16002_010E", "C16002_013E"]                   # more limited-English groups

# build the web address: ask for NAME plus our variables, for every ZIP code area
url = ("https://api.census.gov/data/2024/acs/acs5?get=NAME," + ",".join(vars_)
       + "&for=zip%20code%20tabulation%20area:*")        # "*" means all ZIPs in the country

r = requests.get(url, timeout=120)       # send the request; give up after 120 seconds
print(r.status_code)                     # 200 means it worked

data = r.json()                          # turn the reply into a Python list; the first row holds column names
acs = pd.DataFrame(data[1:], columns=data[0])   # make a table: rows from item 1 on, column names from item 0
print(acs.shape)                         # (number of rows, number of columns)
print(acs.columns.tolist())              # list all column names